# B2-021 — Session 2: Detection Grids and Set Prediction

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260901`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, model hubs, pretrained models, checkpoints, or student data.


## 1. Box coordinates and valid geometry

Use pixel-edge boxes `(x_min,y_min,x_max,y_max)` with x increasing right and y down; intervals are half-open, so width is `x_max-x_min`. Valid boxes satisfy `0<=x_min<x_max<=W` and `0<=y_min<y_max<=H`. Never swap x/y because arrays index `[y,x]`. All fixture boxes are float32 shape `(K,4)` in this convention.

**Checkpoint 1A.** What is the area formula?

**Checkpoint 1B.** Is `(1,2,1,5)` valid?

**Collected answers.** Area is `(x_max-x_min)(y_max-y_min)`; no, it has zero width.


## 2. Intersection, union, and IoU derivation

For boxes A and B, intersection corners are `max` of lower corners and `min` of upper corners. Clamp intersection width and height below at zero. Then `union=area(A)+area(B)-intersection` and `IoU=intersection/union` for positive union. Pairwise broadcasting uses A `(M,1,4)` and B `(1,N,4)` to produce `(M,N)` without Python loops.

**Checkpoint 2A.** Why clamp intersection widths?

**Checkpoint 2B.** What is the IoU range for valid boxes?

**Collected answers.** Disjoint boxes otherwise create a false positive area; IoU lies in `[0,1]`.


## 3. A 4x4 grid target

For image width `W`, height `H`, and an `S×S` grid, let a valid half-open box have center `cx=(x_min+x_max)/2`, `cy=(y_min+y_max)/2`. Its cell is `j=floor(cx*S/W)`, `i=floor(cy*S/H)`. Encode `cx_cell=cx*S/W-j`, `cy_cell=cy*S/H-i`, `w_norm=(x_max-x_min)/W`, `h_norm=(y_max-y_min)/H`; the center offsets are in `[0,1)`. A center exactly on an internal boundary belongs to the cell on its right/below because of `floor`; valid box centers are strictly below `W,H`, so the outer boundary cannot produce index `S`. Store objectness `(S,S)`, class IDs `(S,S)` with `-1` empty, and boxes `(S,S,4)`. If several boxes claim one cell, keep the largest-area box, breaking equal-area ties by the lower source index. The literal fixture has one object per image.

**Checkpoint 3A.** Which index uses x, and what is `cx_cell`?

**Checkpoint 3B.** How is an internal-boundary center assigned?

**Collected answers.** x determines column `j`, and `cx_cell=cx*S/W-j`; `floor` assigns a boundary center to the right/below cell.


## 4. Detection heads and three losses

A tiny detector emits objectness logits `(B,4,4)`, class logits `(B,2,4,4)`, and raw box channels `(B,4,4,4)`. Apply sigmoid to the four box channels to obtain the taught normalized `(cx_cell,cy_cell,w_norm,h_norm)` values. Use `binary_cross_entropy_with_logits` on every objectness cell, `cross_entropy` only at positive cells, and `smooth_l1_loss` only on positive-cell boxes. Combine `L=L_obj+L_cls+2*L_box`, each component reduced by its own mean. Empty images still contribute objectness loss.

**Checkpoint 4A.** Why are box losses masked?

**Checkpoint 4B.** Are sigmoid probabilities passed into BCE-with-logits?

**Collected answers.** Only positive cells have defined boxes; no, pass raw logits.


## 5. Thresholding and class-aware NMS

Decode boxes, compute `score=sigmoid(objectness)*softmax(class_logits)[class]`, discard scores below the pinned threshold, then visit all candidates in stable global descending-score order. Keep a candidate unless a previously kept box of the same class has IoU strictly greater than the NMS threshold. The lower original index resolves equal scores. Different classes do not suppress one another.

**Checkpoint 5A.** Which operation happens first?

**Checkpoint 5B.** Can a class-0 box suppress class 1?

**Collected answers.** Thresholding precedes NMS; no, suppression is class-aware.


## 6. Fixed-seed tiny detection train/eval trace

This actual lesson-only mini-trace uses four literal normalized centers and binary class IDs for training, two distinct held-out centers/classes, and independent linear objectness, two-class, and four-box-channel heads. It uses seed `20260901` and 100 AdamW updates at `lr=.05`. The three heads concatenate conceptually to `(2,1,1,7)`: one objectness, two class, and four normalized box channels; it does not import the p18 fixture. Its loss is exactly `BCE(objectness)+CE(class)+2*SmoothL1(box)`.

| update | train combined loss | held-out combined loss | held-out mean IoU |
|---:|---:|---:|---:|
| 0 | 1.284268 | 1.311272 | 0.032942 |
| 1 | 1.204469 | 1.243969 | 0.044571 |
| 50 | 0.726659 | 0.741738 | 0.270704 |
| 100 | 0.705720 | 0.704910 | 0.564655 |

At update 100, held-out components are objectness BCE `0.008785`, class CE `0.695135`, and box Smooth-L1 `0.000495`; their weighted sum reconstructs the table. The stubborn class component shows why one aggregate number must not hide a weak head. Only training centers/classes enter optimizer steps; held-out rows are decoded under `no_grad`. p18 uses a convolutional 4x4 grid, different steps, and different literal examples, so this is teaching evidence rather than an answer key.

**Checkpoint 6A.** How do the lesson demonstration and p18 update-count/lr protocols differ?

**Checkpoint 6B.** What two behaviors must evaluation certify?

**Collected answers.** The lesson-only demonstration uses exactly 100 updates at `lr=.05`; p18 later requires exactly 16 updates at `lr=.05`. Evaluation checks objectness/loss improvement and decoded-box IoU.


## 7. Worked laboratory: one object through all three heads

Start with the half-open box `(2,1,8,5)` in an image of height 8 and width 12. Its width is 6, height is 4, and area is 24. Its center is `(5,3)`. For a 4x4 grid, x uses width: `5*4/12=5/3`, so column `j=1` and x offset `2/3`. y uses height: `3*4/8=3/2`, so row `i=1` and y offset `1/2`. Normalized width and height are both `1/2`. Writing width and height beside every coordinate prevents the common H/W swap that square images conceal.

The first code cell derives this target and also computes two IoUs. It uses asymmetric boxes and nonsquare geometry deliberately. A buggy program that swaps x and y, indexes the grid as `[j,i]`, or divides x by height may still pass a symmetric 8x8 example. Inspect the complete objectness, class, and box tensors: exactly one objectness cell is one; exactly one class cell is nonnegative; exactly one box row carries the encoding. Empty cells must not contribute class or box loss.

Trace the detector forward as three independent heads from one ReLU stem. Objectness has one logit per cell. Class has two logits per cell. Box has four raw channels per cell and only those channels receive sigmoid before regression. These are seven channels of meaning, not a single seven-class softmax. The combined loss is `BCE_obj + CE_cls + 2*SmoothL1_box`, with each component mean-reduced over its own eligible elements. Omitting the class head can make objectness and IoU look healthy while the detector cannot name an object.

The lesson trace exposes that failure mode: after 100 updates its objectness and box components are small, but class CE remains near chance. Therefore report components as well as the weighted total. When diagnosing p18, first verify shapes and masks, then inspect each held-out component, then decode the best cell and measure IoU and class prediction. A falling combined loss is not by itself a localization or classification certificate.

The second code cell demonstrates stable, class-aware suppression. Threshold first. Within each class, visit descending scores with original index as the tie breaker. Suppress only when IoU is strictly greater than the threshold; equality remains. Candidates from different classes never suppress each other. Keep indices in selection order so the output is deterministic and auditable.

Finally audit information flow. Fixture boxes and labels may build training targets, but held-out boxes and labels may only evaluate after the model is fixed for that checkpoint. Do not choose a threshold, loss weight, or architecture based on the two held-out rows. The exact stored order matters because the certificate and fingerprints bind data identity, target alignment, and optimizer protocol. Reordering has no mathematical effect on a full-batch mean, but accepting it weakens the integrity contract and can hide a substituted or partial batch.

**Checkpoint 7A.** For the nonsquare worked box, which dimensions normalize x and y and which grid cell is positive?

**Checkpoint 7B.** What operation makes the second code cell class-aware NMS rather than thresholding plus sorting?

**Collected answers.** Normalize x by width and y by height, giving positive cell `(i,j)=(1,1)`; after thresholding and stable score ordering, compare IoU against previously kept same-class boxes and suppress only when overlap is strictly above the threshold.


In [ ]:
import torch

box = torch.tensor([2., 1., 8., 5.])
height, width, size = 8, 12, 4
cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
j, i = int(torch.floor(cx * size / width)), int(torch.floor(cy * size / height))
encoding = torch.tensor([cx*size/width-j, cy*size/height-i,
                         (box[2]-box[0])/width, (box[3]-box[1])/height])
assert (i, j) == (1, 1)
assert torch.allclose(encoding, torch.tensor([2/3, 1/2, 1/2, 1/2]), atol=1e-6, rtol=1e-6)

def pairwise_iou_demo(anchor, candidates):
    lower = torch.maximum(anchor[:2], candidates[:, :2])
    upper = torch.minimum(anchor[2:], candidates[:, 2:])
    intersection = (upper - lower).clamp_min(0).prod(dim=1)
    anchor_area = (anchor[2:] - anchor[:2]).prod()
    candidate_area = (candidates[:, 2:] - candidates[:, :2]).prod(dim=1)
    return intersection / (anchor_area + candidate_area - intersection)

didactic_anchor = torch.tensor([0., 0., 5., 4.])
didactic_candidates = torch.tensor([[1., 1., 4., 3.], [4., 0., 7., 4.]])
didactic_ious = pairwise_iou_demo(didactic_anchor, didactic_candidates)
assert torch.allclose(didactic_ious, torch.tensor([.3, 1/7]), atol=1e-6, rtol=1e-6)
print((i, j), encoding, didactic_ious)


In [ ]:
import torch

def class_aware_greedy_nms_demo(boxes, scores, labels, score_threshold=.4, iou_threshold=.5):
    eligible = torch.where(scores >= score_threshold)[0].tolist()
    order = sorted(eligible, key=lambda index: (-float(scores[index]), index))
    kept = []
    for candidate in order:
        suppressed = False
        for kept_index in kept:
            pair_iou = float(pairwise_iou_demo(boxes[kept_index], boxes[candidate].unsqueeze(0))[0])
            if labels[candidate] == labels[kept_index] and pair_iou > iou_threshold:
                suppressed = True
                break
        if not suppressed:
            kept.append(candidate)
    return torch.tensor(kept, dtype=torch.int64), torch.tensor(order, dtype=torch.int64)

nms_boxes = torch.tensor([[0.,0.,4.,4.], [.2,0.,4.2,4.], [0.,0.,4.,4.], [6.,0.,8.,2.], [5.,5.,7.,7.]])
nms_scores = torch.tensor([.74, .91, .83, .29, .68])
nms_labels = torch.tensor([0, 0, 1, 0, 0])
nms_kept, nms_threshold_order = class_aware_greedy_nms_demo(nms_boxes, nms_scores, nms_labels)
assert nms_threshold_order.tolist() == [1, 2, 0, 4]
assert nms_kept.tolist() == [1, 2, 4]
assert 0 not in nms_kept.tolist() and 2 in nms_kept.tolist()
print('threshold/order:', nms_threshold_order.tolist(), 'after class-aware NMS:', nms_kept.tolist())


## 8. Common pitfalls and Exam connections

Common pitfalls are x/y swaps, inclusive-edge area, unclamped intersections, class loss on empty cells, suppressing across classes, and tuning on held-out rows. Exam connections use a score table: identify false positives from low objectness versus wrong class, and duplicates from missing or incorrectly ordered NMS.

**Checkpoint 8A.** Which practice tests coordinate order?

**Checkpoint 8B.** Which practice diagnoses duplicates?

**Collected answers.** p08 tests coordinates; p22 diagnoses score-table failures.


## 9. Forward-only deeper topic: set prediction

Forward-only deeper topic: DETR-like systems predict an unordered set and use bipartite matching instead of a fixed cell assignment. Cross-attention lets learned object queries read image tokens. Matching costs and no-object weighting are not assessed here; B2-021's executable detector remains the explicit 4x4 grid so every target and loss location is auditable.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Is Hungarian matching required?

**Checkpoint 9B.** What target encoding remains executable?

**Collected answers.** No; the 4x4 grid/cell encoding remains required.
